## **Dependencias**

In [ ]:
%pip install -q networkx

## **Imports**

In [ ]:
# === IMPORTS ===
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
from skimage import util, io, data, transform
from skimage.filters import threshold_otsu, threshold_multiotsu
from scipy.ndimage import convolve, generic_filter, label as etiquetar_componentes
from skimage.feature import canny
import networkx as nx
from itertools import combinations

## **Carga de Imágenes**

In [ ]:
#@title Descargar las imagenes { display-mode: "form" }
import io as _io
import os
import urllib.request
import zipfile

REPOSITORIO_URL = "https://github.com/Elementos-Reconocimiento-Visual-UBA/PracticasERV_2026_4toBim"  #@param {type:"string"}
RAMA = "main"  #@param {type:"string"}
CARPETA_EN_REPOSITORIO = "Laboratorio_2_Segmentacion/imagenes"  #@param {type:"string"}
CARPETA_IMAGENES = "imagenes"  # carpeta local donde quedan las imagenes

def descargar_carpeta_de_github(url_zip, carpeta_en_repositorio, carpeta_local):
  # El zip de GitHub tiene todo dentro de una carpeta raiz: <repo>-<rama>/<carpeta_en_repositorio>/archivo
  with urllib.request.urlopen(url_zip) as respuesta:
    zip_repositorio = zipfile.ZipFile(_io.BytesIO(respuesta.read()))

  os.makedirs(carpeta_local, exist_ok=True)
  for nombre in zip_repositorio.namelist():
    # Sacamos la carpeta raiz y nos quedamos con los archivos que estan directamente en la carpeta pedida
    ruta = nombre.split("/", 1)[1] if "/" in nombre else ""
    carpeta, _, archivo = ruta.rpartition("/")
    if carpeta == carpeta_en_repositorio.strip("/") and archivo != "":
      with open(os.path.join(carpeta_local, archivo), "wb") as salida:
        salida.write(zip_repositorio.read(nombre))

if os.path.isdir(CARPETA_IMAGENES):
  print(f"La carpeta {CARPETA_IMAGENES} ya existe, no se descarga nada")
else:
  descargar_carpeta_de_github(f"{REPOSITORIO_URL.rstrip('/')}/archive/refs/heads/{RAMA}.zip", CARPETA_EN_REPOSITORIO, CARPETA_IMAGENES)
  print(f"Descargadas {len(os.listdir(CARPETA_IMAGENES))} imagenes en {CARPETA_IMAGENES}")

## **UMBRALIZACION**

### **Idea**

El objetivo del **umbralizar** es segmentar una imagen en regiones asignando cada píxel a una de dos clases según su nivel de intensidad.

Dado un umbral $T$, cada píxel $f(x,y)$ se clasifica como:

$$
g(x,y)=
\begin{cases}
1 & \text{si } f(x,y) > T\\
0 & \text{si } f(x,y) \le T
\end{cases}
$$

La principal dificultad consiste en elegir un valor adecuado para $T$. Si los objetos y el fondo presentan distribuciones de intensidad bien separadas, es posible utilizar un único umbral para toda la imagen (**global thresholding**).

En la práctica, el valor óptimo del umbral suele variar de una imagen a otra, por lo que resulta útil contar con algoritmos que lo estimen automáticamente.

In [ ]:
def img_mitadymitad():
  img = np.zeros((512,512), dtype=np.uint8)

  img[:, :256] = 95 # mitad izquierda gris oscuro
  img[:, 256:] = 190 # mitad derecha gris claro

  # agregar un poco de ruido
  img = img + np.random.normal(0, 7, img.shape)
  img = np.clip(img, 0, 255).astype(np.uint8)
  return img

def mostrar_threshold(imagen, T=128):

  img_bin = imagen > T

  fig, ax = plt.subplots(1, 3, figsize=(15, 4))

  ax[0].imshow(imagen, cmap="gray")
  ax[0].set_title("Imagen original")

  # Recuadro de la imagen
  ax[0].axis("on")
  ax[0].set_xticks([])
  ax[0].set_yticks([])

  ax[1].hist(imagen.ravel(), bins=256, range=(0,255))
  ax[1].axvline(T, color='r', linewidth=2)
  ax[1].set_title(f"Histograma (T={T})")
  ax[1].set_xlim(0,255)

  ax[2].imshow(img_bin, cmap="gray")
  ax[2].set_title("Imagen umbralizada")

  # Recuadro de la imagen
  ax[2].axis("on")
  ax[2].set_xticks([])
  ax[2].set_yticks([])

  plt.tight_layout()
  plt.show()

In [ ]:
# ELEGIR IMAGEN
camera_man = data.camera()
mostrar_threshold_camera = lambda T : mostrar_threshold(camera_man, T)
interact(
  mostrar_threshold_camera,
  T=IntSlider(
    min=0,
    max=255,
    step=1,
    value=128,
    description='T'
  )
);

In [ ]:
# ELEGIR IMAGEN
mitad_y_mitad = img_mitadymitad()
mostrar_threshold_2_clases = lambda T : mostrar_threshold(mitad_y_mitad, T)
interact(
  mostrar_threshold_2_clases,
  T=IntSlider(
    min=0,
    max=255,
    step=1,
    value=128,
    description='T'
  )
);

### **Umbralización Global Iterativa**

Una forma sencilla de estimar el umbral consiste en asumir que la imagen está compuesta por dos grupos de píxeles:

- Fondo.
- Objetos.

El algoritmo busca un umbral que quede aproximadamente en el medio de las intensidades medias de ambos grupos.

#### Algoritmo

1. Elegir una estimación inicial del umbral $T$.
   - Una elección común es la intensidad media de toda la imagen.

2. Dividir los píxeles en dos grupos:
   - $G_1$: píxeles con intensidad mayor que $T$.
   - $G_2$: píxeles con intensidad menor o igual que $T$.

3. Calcular las medias de intensidad de cada grupo:
   - $m_1$: media de $G_1$.
   - $m_2$: media de $G_2$.

4. Actualizar el umbral:

$$
T_{\text{nuevo}}=\frac{m_1+m_2}{2}
$$

5. Repetir los pasos 2 a 4 hasta que el cambio entre dos iteraciones consecutivas sea menor que una tolerancia $\Delta T$.

In [ ]:
def calcular_nuevo_umbral(imagen, T):
  #Separamos la imagen  en dos, (> T) y (<= T) 
  G1 = #COMPLETAR
  G2 = #COMPLETAR

  # Calculamos las medias
  m1 = #COMPLETAR
  m2 = #COMPLETAR

  # Calculamos el nuevo umbral
  return #COMPLETAR

def thresholding_iterativo(imagen, delta_T=0.5, devolver_iteraciones=False):

  # Aca guardamos las diferentes segmentacions y umbrales que vamos consiguiendo en las iteraciones
  # Las segmentaciones son las mascaras booleanas de las segmentaciones 
  segmentaciones = []
  umbrales = []

  # Calculamos la estimación inicial
  T = imagen.mean()
  while True:
    # COMPLETAR
    # Calculamos el nuevo umbral en relacion al T actual
    # Para verificar si convergue usamos abs(T_nuevo - T) < delta_T
    pass
      

  if(devolver_iteraciones):
    return segmentaciones, umbrales
  else:
    return umbrales[-1]

def graficar_thresholding_iterativo(imagen, delta_T=0.5):
  segmentaciones, umbrales = thresholding_iterativo(imagen, delta_T, devolver_iteraciones=True)
  print(f"Convergió en {len(segmentaciones)} iteraciones.")
  print(f"Umbral final: {umbrales[-1]:.2f}")
  # Mostrar imagen original + todas las iteraciones
  fig, axes = plt.subplots(
    1,
    len(segmentaciones) + 1,
    figsize=(4 * (len(segmentaciones) + 1), 4)
  )

  # Imagen original
  axes[0].imshow(imagen, cmap="gray")
  axes[0].set_title("Original")
  axes[0].axis("off")

  # Iteraciones
  for i, (seg, T) in enumerate(zip(segmentaciones, umbrales), start=1):
    axes[i].imshow(seg, cmap="gray")
    axes[i].set_title(f"Iter {i-1}\nT={T:.2f}")
    axes[i].axis("off")

  plt.tight_layout()
  plt.show()

  print(f"Convergió en {len(segmentaciones)} iteraciones.")
  print(f"Umbral final: {umbrales[-1]:.2f}")

  return umbrales[-1]

### Tests (umbralización iterativa)

In [ ]:
# nombre: (imagen, T, nuevo umbral esperado)
NUEVOS_UMBRALES_ESPERADOS = {
  'grupos_simetricos':     (np.array([10, 10, 30, 30]),         20, 20.0),
  'media_de_las_medias':   (np.array([10, 20, 90]),             15, 32.5),  # la media de toda la imagen daria 40
  'grupos_desparejos':     (np.array([0, 0, 0, 100]),           50, 50.0),  # la media de toda la imagen daria 25
  'pixel_igual_al_umbral': (np.array([10, 20, 30, 40, 50, 60]), 40, 40.0),  # los pixeles iguales a T van al grupo G2
}

# nombre: (imagen, delta_T, umbral de cada iteracion esperado)
UMBRALES_ITERATIVOS_ESPERADOS = {
  'ya_converge':      (np.array([[50] * 5 + [200] * 5] * 4, dtype=np.uint8), 0.5, [125.0]),
  'dos_iteraciones':  (np.array([10] * 6 + [20] * 2 + [90] * 2, dtype=np.uint8), 0.5, [28.0, 51.25]),
}

# nombre: (imagen, umbral final esperado con delta_T = 0.3)
IMAGENES_ITERATIVO = {
  'camera': (data.camera(), 103.07),
  'cell':   (data.cell(),   121.97),
}
DELTA_T_IMAGENES = 0.3

def separa_las_mitades(umbral, imagen):
  # img_mitadymitad tiene la mitad oscura a la izquierda y la clara a la derecha
  mitad = imagen.shape[1] // 2
  izquierda_sobre_umbral = (imagen[:, :mitad] > umbral).mean()
  derecha_sobre_umbral = (imagen[:, mitad:] > umbral).mean()
  return izquierda_sobre_umbral < 0.001 and derecha_sobre_umbral > 0.999

def verificar_nuevo_umbral(nombre):
  imagen, T, esperado = NUEVOS_UMBRALES_ESPERADOS[nombre]
  obtenido = calcular_nuevo_umbral(imagen, T)

  assert np.isclose(obtenido, esperado), f"[{nombre}] se esperaba {esperado} pero se obtuvo {obtenido}"
  print(f"OK [{nombre}] nuevo umbral = {obtenido}")

def verificar_iterativo(nombre):
  imagen, delta_T, esperados = UMBRALES_ITERATIVOS_ESPERADOS[nombre]
  segmentaciones, umbrales = thresholding_iterativo(imagen, delta_T, devolver_iteraciones=True)
  umbral_final = thresholding_iterativo(imagen, delta_T)

  assert np.allclose(umbrales, esperados), f"[{nombre}] se esperaban los umbrales {esperados} pero se obtuvo {np.round(umbrales, 2).tolist()}"
  assert len(segmentaciones) == len(umbrales), f"[{nombre}] tiene que haber una segmentacion por cada umbral"
  assert all(np.array_equal(segmentacion, imagen > umbral) for segmentacion, umbral in zip(segmentaciones, umbrales)), f"[{nombre}] cada segmentacion tiene que ser imagen > umbral de su iteracion"
  assert np.isclose(umbral_final, esperados[-1]), f"[{nombre}] sin devolver_iteraciones se esperaba el umbral final {esperados[-1]} pero se obtuvo {umbral_final}"
  print(f"OK [{nombre}] {len(umbrales)} iteraciones, umbral final = {umbral_final}")

def verificar_iterativo_imagen(nombre):
  imagen, esperado = IMAGENES_ITERATIVO[nombre]
  umbral_final = thresholding_iterativo(imagen, DELTA_T_IMAGENES)

  assert np.isclose(umbral_final, esperado, atol=0.01), f"[{nombre}] se esperaba un umbral final de {esperado} pero se obtuvo {umbral_final:.2f}"
  # En el umbral final, una iteracion mas ya no lo mueve mas que delta_T
  assert abs(calcular_nuevo_umbral(imagen, umbral_final) - umbral_final) < DELTA_T_IMAGENES, f"[{nombre}] el umbral final todavia se mueve mas que delta_T"
  print(f"OK [{nombre}] umbral final = {umbral_final:.2f}")

def verificar_iterativo_mitad_y_mitad():
  imagen = img_mitadymitad()
  umbral_final = thresholding_iterativo(imagen, DELTA_T_IMAGENES)

  assert separa_las_mitades(umbral_final, imagen), f"el umbral {umbral_final:.2f} no separa la mitad oscura (95) de la clara (190)"
  print(f"OK [mitad_y_mitad] umbral final = {umbral_final:.2f}")

### Verificacion calcular_nuevo_umbral

In [ ]:
for nombre in NUEVOS_UMBRALES_ESPERADOS:
  verificar_nuevo_umbral(nombre)

### Verificacion thresholding_iterativo

In [ ]:
for nombre in UMBRALES_ITERATIVOS_ESPERADOS:
  verificar_iterativo(nombre)

In [ ]:
for nombre in IMAGENES_ITERATIVO:
  verificar_iterativo_imagen(nombre)

verificar_iterativo_mitad_y_mitad()

#### **Ejemplos**

In [ ]:
img = data.cell()

umbral_global = graficar_thresholding_iterativo(img, delta_T=0.3)
mostrar_threshold(img, umbral_global)

In [ ]:
img = data.camera()

umbral_global = graficar_thresholding_iterativo(img, delta_T=0.3)
mostrar_threshold(img, umbral_global)

### **Umbralización global óptima con el método de Otsu**

Otsu propone una forma automática de elegir el umbral $T$: el algoritmo prueba todos los posibles valores de umbral $k$ y elige aquel que maximiza la separación estadística entre dos clases de píxeles.

Sea una imagen con $L$ niveles de gris:

$$
\{0,1,2,\dots,L-1\}
$$

y sea $n_i$ la cantidad de píxeles con intensidad $i$. A partir de esto se construye el histograma normalizado:

$$
p_i = \frac{n_i}{MN}
$$

donde $MN$ es la cantidad total de píxeles de la imagen.

Para cada posible umbral $k$, se divide el histograma en dos clases:

$$
C_1 = [0,1,\dots,k]
$$

$$
C_2 = [k+1,k+2,\dots,L-1]
$$

La probabilidad acumulada de la primera clase es:

$$
P_1(k) = \sum_{i=0}^{k} p_i
$$

y la probabilidad de la segunda clase es:

$$
P_2(k) = 1 - P_1(k)
$$

La media global de la imagen sera:

$$
m_G = \sum_{i=0}^{L-1} i p_i
$$

Notando como $m_1$ y $m_2$ a las medias de la clases 1 y 2 respectivamente, el criterio de Otsu consiste en calcular, para cada $k$, la varianza entre clases:

$$
\sigma_B^2(k) = P_1(m_1 - m_G)^2 + P_2(m_2 - m_G)^2
$$

El umbral óptimo $k^*$ es aquel para el cual la varianza entre clases alcanza su valor máximo:

$$
\sigma_B^2(k^*) = \max_{0 \leq k \leq L-1} \sigma_B^2(k)
$$

Es decir, se evalúa $\sigma_B^2(k)$ para todos los posibles umbrales $k$, y se selecciona como $k^*$ aquel valor que produce la mayor separación entre clases.

Una vez obtenido $k^*$, se aplica el threshold sobre la imagen original:

$$
g(x,y)=
\begin{cases}
255 & \text{si } f(x,y) > k^* \\
0 & \text{si } f(x,y) \leq k^*
\end{cases}
$$






Esta definicion tiene un problema, ya que tendremos que recalcular $m_1$ y $m_2$ para cada $k$ ¿Podemos mejorar esto?

Veamos que $m_1$ puede escribirse como:

$$
m_1 = \sum_{i=0}^{k} i \cdot p_1(i)
$$

donde $p_1(i)$ es la probabilidad de que muestrar a i en la clase 1, la cual podemos reescribir como una probabilidad condicional

$$
p_1(i) = P(i|C_1) = \frac{P(C_1|i) ⋅ p_i}{P_1}
$$

Por lo que podemos definir a $m_1$ segun el promedio acumulado hasta k

$$
m_1 = \sum_{i=0}^{k} i ⋅ \frac{p_i}{P_1} = \frac{1}{P_1}\sum_{i=0}^{k} i ⋅ p_i  
$$

Llamemos $m(k)$ al promedio acumulado hasta k

$$
m(k) = \sum_{i=0}^{k} i ⋅ p_i
$$

Finalmente

$$
m_1 = \frac{m(k)}{P_1}
$$

Luego usaremos que
$$
m_G = P_1 \cdot m_1 + (1- P_2) \cdot m_2
$$
por lo que
$$
m_2 = \frac{m_G - P_1 \cdot m_1}{1-P_2} = \frac{m_G - m(k)}{1-P_2}
$$
\
Sustituyendo en $\sigma_B^2(k)$ y desarrollando nos queda finalmente:

\
$$
\sigma_B^2(k) =
\frac{[m_G P_1(k) - m(k)]^2}
{P_1(k)[1-P_1(k)]}
$$

Esta definicion nos permite construi $P_1$ y $m(k)$ mientras testeamos a $k$

In [ ]:
def otsu_manual(imagen, graficar=False):

  # Histograma de 256 niveles de gris
  hist, _ = np.histogram(imagen, 256, range=(0, 255), density=True)
  p = hist / hist.sum()
  niveles = np.arange(256)

  # Calculamos el promedio global de la imagen
  mG = # COMPLETAR

  # Varianza entre clases
  sigma_b2 = np.zeros(256)
  P1 = 0
  m = 0

  for k in range(256):
    # Actualizamos P1 y m
    # COMPLETAR
    
    # Cuando uno de los grupos esta vacio, la varianza entre clases no tiene sentido y se indefine
    if P1 == 0 or P1 == 1:
      continue

    # Calculamos el sigma para esta k
    sigma_b2[k] = # COMPLETAR

  # Threshold óptimo: k que maximiza sigma_b2

  # Encontramos el k que causo el maximo sigma
  maximos = np.where(sigma_b2 == np.max(sigma_b2))[0]
  T_otsu = int(np.mean(maximos))  # si hay varios máximos, promediamos

  if graficar:
    return T_otsu, hist, sigma_b2
  return T_otsu

### Tests (Otsu)

In [ ]:
IMAGENES_OTSU = {
  'camera': data.camera(),
  'coins':  data.coins(),
  'page':   data.page(),
  'cell':   data.cell(),
}

# Aceptamos 1 de diferencia contra skimage porque agrupa los niveles de gris un poco distinto
TOLERANCIA_OTSU = 1

def verificar_otsu(nombre):
  imagen = IMAGENES_OTSU[nombre]
  esperado = threshold_otsu(imagen)
  obtenido = otsu_manual(imagen)

  assert abs(obtenido - esperado) <= TOLERANCIA_OTSU, f"[{nombre}] skimage da {esperado} pero se obtuvo {obtenido}"
  print(f"OK [{nombre}] T = {obtenido} (skimage: {esperado})")

def verificar_otsu_dos_niveles():
  # Con solo dos niveles de gris cualquier umbral entre ellos separa perfecto
  imagen = np.array([[50] * 5 + [200] * 5] * 4, dtype=np.uint8)
  umbral = otsu_manual(imagen)

  assert np.array_equal(imagen > umbral, imagen == 200), f"el umbral {umbral} no separa los niveles 50 y 200"
  print(f"OK [dos_niveles] T = {umbral}")

def verificar_otsu_mitad_y_mitad():
  imagen = img_mitadymitad()
  umbral = otsu_manual(imagen)

  assert separa_las_mitades(umbral, imagen), f"el umbral {umbral} no separa la mitad oscura (95) de la clara (190)"
  print(f"OK [mitad_y_mitad] T = {umbral}")

def verificar_otsu_graficar():
  imagen = IMAGENES_OTSU['camera']
  umbral, histograma, varianza_entre_clases = otsu_manual(imagen, graficar=True)

  assert histograma.shape == (256,) and varianza_entre_clases.shape == (256,), "el histograma y la varianza entre clases tienen que tener un valor por nivel de gris"
  assert np.isclose(varianza_entre_clases[umbral], varianza_entre_clases.max()), "el umbral tiene que ser donde la varianza entre clases es maxima"
  print(f"OK [graficar] T = {umbral}")

### Verificacion Otsu

In [ ]:
for nombre in IMAGENES_OTSU:
  verificar_otsu(nombre)

In [ ]:
verificar_otsu_dos_niveles()
verificar_otsu_mitad_y_mitad()
verificar_otsu_graficar()

#### **Ejemplo**

In [ ]:
imagenOriginal = data.camera()
T_otsu, hist, sigma_b2 = otsu_manual(imagenOriginal, graficar=True)

# Aplicar threshold
imagenOtsu = np.zeros_like(imagenOriginal)
imagenOtsu[imagenOriginal > T_otsu] = 255
imagenOtsu[imagenOriginal <= T_otsu] = 0

# Mostrar resultados
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

axs[0].imshow(imagenOriginal, cmap='gray', clim=(0, 255))
axs[0].set_title("Imagen original")
axs[0].axis("off")

axs[1].hist(imagenOriginal.ravel(), bins=256, range=(0,255))
axs[1].axvline(T_otsu, color='r', linewidth=2)
axs[1].set_title(f"Histograma (T={T_otsu})")

axs[2].imshow(imagenOtsu, cmap='gray', clim=(0, 255))
axs[2].set_title("Otsu")
axs[2].axis("off")

plt.tight_layout()
plt.show()

print("Threshold de Otsu:", T_otsu)

#### **Comparación contra método iterativo**

In [ ]:
# img = data.cell()

# Otras opciones para analizar!!!
img = util.img_as_ubyte(io.imread("./imagenes/polymercell.tif", as_gray=True))
# img = data.clock()

# Umbrales
T_iterativo = thresholding_iterativo(img, delta_T=0.3)
T_otsu = otsu_manual(img)

# Segmentaciones
seg_iterativa = img > T_iterativo
seg_otsu = img > T_otsu

# Visualización
fig, ax = plt.subplots(2, 2, figsize=(12, 10))

# Imagen original
ax[0,0].imshow(img, cmap="gray")
ax[0,0].set_title("Imagen original")
ax[0,0].axis("off")

# Histograma
ax[0,1].hist(img.ravel(), bins=256, range=(0,255))
ax[0,1].axvline(T_iterativo, color='r', linewidth=2,
                label=f'Iterativo = {T_iterativo:.1f}')
ax[0,1].axvline(T_otsu, color='g', linewidth=2,
                label=f'Otsu = {T_otsu:.1f}')
ax[0,1].set_title("Histograma")
ax[0,1].set_xlim(0,255)
ax[0,1].legend()

# Resultado iterativo
ax[1,0].imshow(seg_iterativa, cmap="gray")
ax[1,0].set_title(f"Thresholding Iterativo\nT = {T_iterativo:.2f}")
ax[1,0].axis("off")

# Resultado Otsu
ax[1,1].imshow(seg_otsu, cmap="gray")
ax[1,1].set_title(f"Otsu\nT = {T_otsu:.2f}")
ax[1,1].axis("off")

plt.tight_layout()
plt.show()

#### **Limitaciones**

##### **Mejora del Thresholding global mediante suavizado**

Cuando una imagen está afectada por ruido, los niveles de gris del objeto y del fondo pueden superponerse en el histograma, dificultando la selección de un umbral adecuado. En estas condiciones, métodos automáticos como el umbral de Otsu pueden producir segmentaciones deficientes.

Una estrategia simple consiste en aplicar previamente un filtro de suavizado (por ejemplo, un filtro gausiano). El suavizado reduce las fluctuaciones producidas por el ruido y hace que las distribuciones de intensidad correspondientes al objeto y al fondo se separen mejor en el histograma. Como consecuencia, el umbral calculado por Otsu suele producir una segmentación mucho más precisa.

In [ ]:
# Leer imagen original
img = io.imread("./imagenes/blobs.tif").astype(np.float32)

# Agregar ruido
sigma = 50

img_ruidosa = img + sigma * np.random.randn(*img.shape)

# Umbralizado de OTSU sobre imagen ruidosa
t1 = threshold_otsu(img_ruidosa)
bin_ruidosa = img_ruidosa > t1


# Suavizado (filtro promedio)
# Aplicamos un filtro de promedio de tamaño 7x7
kernel = np.ones((7,7), dtype=np.float32) / 49

img_suavizada = convolve(
    img_ruidosa,
    kernel,
    mode='reflect'
)

# Umbralizado de OTSU sobre imagen suavizada
t2 = threshold_otsu(img_suavizada)
bin_suavizada = img_suavizada > t2

# Mostrar resultados
fig, ax = plt.subplots(2, 3, figsize=(12, 6))

# Imagen ruidosa
ax[0,0].imshow(np.clip(img_ruidosa, 0, 255), cmap='gray', vmin=0, vmax=255)
#ax[0,0].imshow(img_ruidosa, cmap='gray', vmin=0, vmax=1)
ax[0,0].set_title("Imagen con ruido")
ax[0,0].axis("off")

# Histograma imagen ruidosa
ax[0,1].hist(img_ruidosa.ravel(), bins=256, range=(0,255))
ax[0,1].set_title("Histograma")
ax[0,1].axvline(t1, linestyle='--', color='red')

# Resultado Otsu
ax[0,2].imshow(bin_ruidosa, cmap='gray')
ax[0,2].set_title("Otsu sobre imagen ruidosa")
ax[0,2].axis("off")

# Imagen suavizada
ax[1,0].imshow(np.clip(img_suavizada, 0, 255), cmap='gray', vmin=0, vmax=255)
#ax[1,0].imshow(img_suavizada, cmap='gray', vmin=0, vmax=1)
ax[1,0].set_title("Imagen suavizada")
ax[1,0].axis("off")

# Histograma de imagen suavizada
ax[1,1].hist(img_suavizada.ravel(), bins=256, range=(0,255))
ax[1,1].set_title("Histograma")
ax[1,1].axvline(t2, linestyle='--', color='red')

# Resultado final
ax[1,2].imshow(bin_suavizada, cmap='gray')
ax[1,2].set_title("Otsu tras suavizado")
ax[1,2].axis("off")

plt.tight_layout()
plt.show()

print(f"Umbral Otsu (ruidosa):    {t1:.4f}")
print(f"Umbral Otsu (suavizada): {t2:.4f}")

## **Segmentacion**

### **Segmentación por cortes de grafos (Graph Cuts)**

Umbralizar decide cada píxel **por separado** mirando únicamente su intensidad: no le importa qué hay a su alrededor. Por eso es sensible al ruido y no distingue dos objetos que tengan el mismo brillo.

La idea de esta sección es modelar la imagen como un **grafo** y segmentarla **cortándolo en dos partes**: *objeto* y *fondo*. Vamos a armar el grafo de forma que el corte más barato pase justo por donde los píxeles vecinos son más distintos entre sí, es decir, por los bordes del objeto.

#### **Redes de flujo**

Una **red de flujo** es un grafo dirigido $G=(V,E)$ donde cada arista $(u,v)$ tiene una **capacidad** $c(u,v) \ge 0$, y donde hay dos vértices especiales: la **fuente** $s$ y el **sumidero** $t$.

Un **flujo** es una función $f$ sobre las aristas que cumple:

- **Restricción de capacidad:** por una arista no puede pasar más que su capacidad.

$$
0 \le f(u,v) \le c(u,v)
$$

- **Conservación del flujo:** en todo vértice que no sea $s$ ni $t$, lo que entra es igual a lo que sale.

$$
\sum_{v \in V} f(v,u) = \sum_{v \in V} f(u,v) \qquad \forall u \in V \setminus \{s,t\}
$$

El **valor del flujo** es lo que sale neto de la fuente:

$$
|f| = \sum_{v \in V} f(s,v) - \sum_{v \in V} f(v,s)
$$

Un **corte $s$-$t$** es una partición de los vértices en dos conjuntos disjuntos $S$ y $T$, con $s \in S$ y $t \in T$. Su **capacidad** es la suma de las capacidades de las aristas que van de $S$ hacia $T$:

$$
c(S,T) = \sum_{u \in S,\; v \in T} c(u,v)
$$

Las aristas que van de $T$ hacia $S$ **no** cuentan.

##### **Teorema de max-flow min-cut**

El valor del flujo máximo que se puede mandar de $s$ a $t$ es igual a la capacidad del corte mínimo:

$$
\max_f |f| = \min_{(S,T)} c(S,T)
$$

Entonces, para encontrar el corte de menor capacidad alcanza con resolver un problema de flujo máximo. **No vamos a ver cómo se resuelve** (Ford-Fulkerson, Edmonds-Karp, push-relabel...): vamos a usar una librería que ya lo hace.

Veamos el teorema en una red chica. Probamos **todos** los cortes posibles (son pocos) y comparamos el mejor contra el flujo máximo que calcula `networkx`.

In [ ]:
def dibujar_red(red, posiciones):
  capacidades = nx.get_edge_attributes(red, "capacity")
  plt.figure(figsize=(6, 4))
  nx.draw_networkx(red, posiciones, node_color="lightgray", edgecolors="black", node_size=900, arrowsize=20)
  nx.draw_networkx_edge_labels(red, posiciones, edge_labels=capacidades, font_size=12, rotate=False)
  plt.axis("off")
  plt.show()

def capacidad_del_corte(red, conjunto_s):
  # Solo cuentan las aristas que van de S a T
  return sum(capacidad for u, v, capacidad in red.edges(data="capacity") if u in conjunto_s and v not in conjunto_s)

red = nx.DiGraph()
red.add_edge("s", "a", capacity=3)
red.add_edge("s", "b", capacity=2)
red.add_edge("a", "b", capacity=1)
red.add_edge("a", "t", capacity=2)
red.add_edge("b", "t", capacity=3)

# El numero sobre cada flecha es su capacidad
dibujar_red(red, {"s": (0, 0), "a": (1, 1), "b": (1, -1), "t": (2, 0)})

vertices_intermedios = ["a", "b"]
for cantidad in range(len(vertices_intermedios) + 1):
  for subconjunto in combinations(vertices_intermedios, cantidad):
    conjunto_s = {"s", *subconjunto}
    print(f"S = {sorted(conjunto_s)}  ->  capacidad del corte = {capacidad_del_corte(red, conjunto_s)}")

print(f"Flujo maximo = {nx.maximum_flow_value(red, 's', 't')}")

#### **De imagen a grafo**

Dada una imagen de $alto \times ancho$ píxeles, el grafo tiene:

**Vértices**

- Un vértice por cada píxel. Los numeramos recorriendo la imagen por filas: el píxel `(fila, columna)` es el vértice `fila * ancho + columna`.
- La fuente $s$ y el sumidero $t$, que no se corresponden con ningún píxel.

**n-links** (*neighbor links*): conectan cada píxel con sus 4 vecinos (arriba, abajo, izquierda y derecha). Queremos que sea **caro cortar entre píxeles parecidos** y **barato cortar entre píxeles muy distintos**. Para eso usamos la **penalidad de frontera** entre los píxeles $p$ y $q$:

$$
B(I_p, I_q) = 100 \cdot \exp\left(-\frac{(I_p - I_q)^2}{2\sigma^2}\right)
$$

donde $I_p$ es la intensidad del píxel $p$. Si $|I_p - I_q|$ es chico, $B$ vale casi $100$, si es grande, $B$ es casi $0$. El $\sigma$ define qué tan grande tiene que ser una diferencia para considerarla un borde (usaremos $\sigma = 30$). Se pasa a **entero** porque la capacidad debe ser entera.

Cada n-link se agrega **en los dos sentidos** con la misma capacidad, porque el corte solo cuenta las aristas que van de $S$ a $T$.

**t-links** (*terminal links*): conectan los píxeles con $s$ y $t$. Los vamos a crear segun unas **semillas**, píxeles que sabemos de qué lado están.

- Cada semilla de **fondo** $p$ tiene una arista $s \to p$ con capacidad $K$.
- Cada semilla de **objeto** $p$ tiene una arista $p \to t$ con capacidad $K$.

$K$ es una capacidad más grande que la suma de las aristas de cualquier píxel, así que nunca conviene cortar una semilla. Ya les damos calculada la función que obtiene $K$.

Con esta convención, **después de cortar, el lado de $s$ es el fondo y el lado de $t$ es el objeto**.

Para que el solver termine rápido usamos una imagen chica: el grafo tiene un vértice por píxel y cuatro aristas por vértice, así que crece con la cantidad de píxeles. Con 64x64 son 4096 vértices y `networkx` tarda alrededor de un segundo.

Las semillas se dan como máscaras booleanas del mismo tamaño que la imagen.

In [ ]:
def cargar_imagen_monedas(lado=64):
  # Recortamos las dos filas completas de monedas y achicamos para que el grafo sea chico
  recorte = data.coins()[10:180, 40:210]
  return util.img_as_ubyte(transform.resize(recorte, (lado, lado), anti_aliasing=True))

def crear_semillas(forma, rectangulos):
  # rectangulos: lista de (fila_inicio, fila_fin, columna_inicio, columna_fin)
  semillas = np.zeros(forma, dtype=bool)
  for fila_inicio, fila_fin, columna_inicio, columna_fin in rectangulos:
    semillas[fila_inicio:fila_fin, columna_inicio:columna_fin] = True
  return semillas

imagen_monedas = cargar_imagen_monedas()

# Una mancha en el centro de cada moneda
SEMILLAS_OBJETO = crear_semillas(imagen_monedas.shape, [
  (12, 19, 40, 47), (12, 19, 57, 64), (13, 20, 0, 7), (14, 21, 19, 26),
  (39, 46, 0, 7),   (39, 46, 56, 63), (40, 47, 20, 27), (40, 47, 39, 46),
])

# Una franja de fondo arriba, otra entre las dos filas de monedas y otra abajo
SEMILLAS_FONDO = crear_semillas(imagen_monedas.shape, [(0, 3, 0, 64), (27, 31, 0, 64), (52, 56, 0, 64)])

In [ ]:
def dibujar_semillas(ax, imagen, semillas_fondo, semillas_objeto):
  imagen_rgb = np.dstack([imagen] * 3)
  imagen_rgb[semillas_fondo] = (0, 0, 255)
  imagen_rgb[semillas_objeto] = (255, 0, 0)
  ax.imshow(imagen_rgb)
  ax.set_title("Imagen y semillas\n(azul: fondo, rojo: objeto)")
  ax.axis("off")

def mostrar_segmentacion(imagen, semillas_fondo, semillas_objeto, segmentacion):
  fig, ax = plt.subplots(1, 2, figsize=(10, 5))

  dibujar_semillas(ax[0], imagen, semillas_fondo, semillas_objeto)

  ax[1].imshow(segmentacion, cmap="gray")
  ax[1].set_title("Segmentacion (blanco: objeto)")
  ax[1].axis("off")

  plt.tight_layout()
  plt.show()

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
dibujar_semillas(ax, imagen_monedas, SEMILLAS_FONDO, SEMILLAS_OBJETO)
plt.show()

### ***Ejercicio***: Construir el grafo de la imagen

Completar las funciones que arman la red de flujo de una imagen usando un `nx.DiGraph` de `networkx`. Los vértices de los píxeles son enteros y la fuente y el sumidero son los strings `FUENTE` y `SUMIDERO`. Las capacidades se guardan en el atributo `capacity` de cada arista (`grafo.add_edge(u, v, capacity=...)`).

- Ejercicio: implementar
    - penalidad_frontera
    - agregar_n_links
    - agregar_t_links

Ya están dadas `id_de_pixel`, `vecinos_derecha_y_abajo`, `calcular_capacidad_semilla` y `construir_grafo`, que llama a las funciones anteriores.

In [ ]:
FUENTE = "s"
SUMIDERO = "t"

def id_de_pixel(fila, columna, ancho):
  return fila * ancho + columna

def vecinos_derecha_y_abajo(fila, columna, alto, ancho):
  # Solo derecha y abajo: asi cada par de vecinos se visita una sola vez
  vecinos = []
  if columna + 1 < ancho: vecinos.append((fila, columna + 1))
  if fila + 1 < alto: vecinos.append((fila + 1, columna))
  return vecinos

def calcular_capacidad_semilla(grafo):
  # Hay que llamarla despues de agregar los n-links y antes de agregar los t-links
  capacidad_maxima_por_pixel = max(grafo.out_degree(pixel, weight="capacity") for pixel in grafo.nodes)
  return 1 + capacidad_maxima_por_pixel

In [ ]:
def penalidad_frontera(intensidad_p, intensidad_q, sigma=30):
  # B(Ip, Iq) = 100 * exp(-(Ip - Iq)^2 / (2 * sigma^2)), truncada a entero
  # Cuidado: la imagen es uint8 y restar uint8 da la vuelta (3 - 5 = 254). Hay que pasar a float antes de restar.
  # COMPLETAR-

In [ ]:
def agregar_n_links(grafo, imagen, sigma):
  # Agrega al grafo una arista por cada par de pixeles vecinos, en los dos sentidos y con la penalidad de frontera como capacidad
  alto, ancho = imagen.shape

  for fila in range(alto):
    for columna in range(ancho):
      id_pixel = id_de_pixel(fila, columna, ancho)

      # Por el vecino de derecha y abajo
      # Calculamos el id del vecino y la penalidad de frontera
      # Luego creamos el n-link en el grafo para ambos sentidos
      # Para crear el vertice utilizamos:
      # grafo.add_edge(id_origen, id_destino, capacity=capacidad)

In [ ]:
def agregar_t_links(grafo, semillas_fondo, semillas_objeto):
  # Fondo: FUENTE -> pixel. Objeto: pixel -> SUMIDERO. Las dos con capacidad K
  capacidad_semilla = # Calculamos la capacidad K que seguro el algoritmo no cortara

  # np.flatnonzero recorre la mascara por filas, que es justo como numeramos los vertices
  for id_pixel in np.flatnonzero(semillas_fondo):
    # Agregamos el edge FUENTE -> pixel
    pass

  for id_pixel in np.flatnonzero(semillas_objeto):
    # Agregamos el edge pixel -> SUMIDERO
    pass

In [ ]:
def construir_grafo(imagen, semillas_fondo, semillas_objeto, sigma=30):
  if semillas_fondo.shape != imagen.shape or semillas_objeto.shape != imagen.shape:
    raise ValueError("Las semillas tienen que tener la misma forma que la imagen")
  if not semillas_fondo.any() or not semillas_objeto.any():
    raise ValueError("Hace falta al menos una semilla de fondo y una de objeto")

  alto, ancho = imagen.shape
  grafo = nx.DiGraph()
  grafo.add_nodes_from(range(alto * ancho))

  agregar_n_links(grafo, imagen, sigma)
  agregar_t_links(grafo, semillas_fondo, semillas_objeto)
  return grafo

In [ ]:
def mostrar_recorte(imagen, filas, columnas):
  # Marca el recorte sobre la imagen completa y lo muestra ampliado con el numero de vertice de cada pixel
  recorte = imagen[filas, columnas]
  alto, ancho = recorte.shape
  izquierda, derecha = columnas.start - 0.5, columnas.stop - 0.5
  arriba, abajo = filas.start - 0.5, filas.stop - 0.5

  fig, ax = plt.subplots(1, 2, figsize=(10, 5))

  ax[0].imshow(imagen, cmap="gray", vmin=0, vmax=255)
  ax[0].plot([izquierda, derecha, derecha, izquierda, izquierda], [arriba, arriba, abajo, abajo, arriba], color="orange", linewidth=2)
  ax[0].set_title("Imagen completa y recorte")
  ax[0].axis("off")

  ax[1].imshow(recorte, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
  for fila in range(alto):
    for columna in range(ancho):
      ax[1].text(columna, fila, id_de_pixel(fila, columna, ancho), ha="center", va="center", bbox=dict(boxstyle="round", fc="white", alpha=0.8))
  ax[1].set_title("Recorte ampliado\n(numero = vertice del pixel)")
  ax[1].axis("off")

  plt.tight_layout()
  plt.show()

def dibujar_grafo_imagen(grafo, imagen):
  # Cada pixel es un nodo con el color de su intensidad y su numero de vertice. El numero sobre cada linea es su capacidad
  alto, ancho = imagen.shape
  pixeles = list(range(alto * ancho))
  posiciones = {id_de_pixel(fila, columna, ancho): (columna, -fila) for fila in range(alto) for columna in range(ancho)}
  posiciones[FUENTE] = (-1.5, -(alto - 1) / 2)
  posiciones[SUMIDERO] = (ancho + 0.5, -(alto - 1) / 2)

  # Los n-links van en los dos sentidos con la misma capacidad: dibujamos una sola linea por par
  n_links = [(u, v) for u, v in grafo.subgraph(pixeles).edges if u < v]
  t_links_fondo = [(u, v) for u, v in grafo.edges if u == FUENTE]
  t_links_objeto = [(u, v) for u, v in grafo.edges if v == SUMIDERO]
  capacidades = nx.get_edge_attributes(grafo, "capacity")
  capacidades_dibujadas = {arista: capacidades[arista] for arista in n_links + t_links_fondo + t_links_objeto}

  plt.figure(figsize=(9, 7))
  nx.draw_networkx_nodes(grafo, posiciones, nodelist=pixeles, node_color=imagen.ravel(), cmap="gray", vmin=0, vmax=255, edgecolors="black", node_size=900)
  nx.draw_networkx_nodes(grafo, posiciones, nodelist=[FUENTE], node_color="blue", node_size=900)
  nx.draw_networkx_nodes(grafo, posiciones, nodelist=[SUMIDERO], node_color="red", node_size=900)
  nx.draw_networkx_edges(grafo, posiciones, edgelist=n_links, arrows=False)
  nx.draw_networkx_edges(grafo, posiciones, edgelist=t_links_fondo, edge_color="blue", arrowsize=20, node_size=900)
  nx.draw_networkx_edges(grafo, posiciones, edgelist=t_links_objeto, edge_color="red", arrowsize=20, node_size=900)
  nx.draw_networkx_edge_labels(grafo, posiciones, edge_labels=capacidades_dibujadas, font_size=8, rotate=False)
  nx.draw_networkx_labels(grafo, posiciones, labels={pixel: pixel for pixel in pixeles}, font_size=9, bbox=dict(boxstyle="round", fc="white", alpha=0.8))
  nx.draw_networkx_labels(grafo, posiciones, labels={FUENTE: FUENTE, SUMIDERO: SUMIDERO}, font_color="white", font_size=9)
  plt.axis("off")
  plt.show()

Antes de seguir, verifiquemos que el grafo tenga la forma esperada con un recorte de 4x4 píxeles que cruza el borde de una moneda (primero lo mostramos, con el número de vértice de cada píxel): un vértice por píxel más la fuente y el sumidero, dos aristas por cada par de vecinos y una arista por cada semilla.

Además lo dibujamos para tener una referencia visual: cada píxel es un nodo con el color de su intensidad, el número dentro del nodo es su vértice y el número sobre cada línea es su capacidad. Fijense que las capacidades bajan justo donde la imagen pasa de oscuro a claro, que es por donde va a querer cortar el algoritmo. La fuente (azul) se conecta a las semillas de fondo y las semillas de objeto se conectan al sumidero (rojo).

In [ ]:
FILAS_RECORTE = slice(10, 14)
COLUMNAS_RECORTE = slice(16, 20)
imagen_chica = imagen_monedas[FILAS_RECORTE, COLUMNAS_RECORTE]
mostrar_recorte(imagen_monedas, FILAS_RECORTE, COLUMNAS_RECORTE)

semillas_fondo_chicas = crear_semillas(imagen_chica.shape, [(0, 1, 0, 2)])
semillas_objeto_chicas = crear_semillas(imagen_chica.shape, [(3, 4, 2, 4)])
grafo_chico = construir_grafo(imagen_chica, semillas_fondo_chicas, semillas_objeto_chicas)

alto, ancho = imagen_chica.shape
vertices_esperados = alto * ancho + 2
n_links_esperados = 2 * (alto * (ancho - 1) + ancho * (alto - 1))
t_links_esperados = semillas_fondo_chicas.sum() + semillas_objeto_chicas.sum()

print(f"Vertices: {grafo_chico.number_of_nodes()} (esperados {vertices_esperados})")
print(f"Aristas:  {grafo_chico.number_of_edges()} (esperadas {n_links_esperados + t_links_esperados})")

dibujar_grafo_imagen(grafo_chico, imagen_chica)

### Tests (construir el grafo)

In [ ]:
import time

# (intensidad_p, intensidad_q, sigma, penalidad esperada)
PENALIDADES_ESPERADAS = {
  'iguales':          (100, 100, 30, 100),
  'muy_distintos':    (0,   255, 30, 0),
  'diferencia_media': (100, 130, 30, 60),
  'orden_inverso':    (130, 100, 30, 60),
  'sigma_mayor':      (100, 130, 60, 88),
  'uint8':            (np.uint8(3), np.uint8(5), 30, 99),  # restar uint8 da la vuelta: 3 - 5 = 254
}

# Imagen de 2x3, los vertices son [[0, 1, 2], [3, 4, 5]]. Entre la columna 1 y la 2 hay un borde muy marcado
IMAGEN_TEST_GRAFO = np.array([[0, 0, 200], [0, 0, 200]], dtype=np.uint8)
SEMILLAS_FONDO_TEST_GRAFO = crear_semillas(IMAGEN_TEST_GRAFO.shape, [(0, 2, 0, 1)])   # vertices 0 y 3
SEMILLAS_OBJETO_TEST_GRAFO = crear_semillas(IMAGEN_TEST_GRAFO.shape, [(0, 2, 2, 3)])  # vertices 2 y 5

# Capacidad esperada de cada par de vecinos, que tiene que estar en los dos sentidos
N_LINKS_ESPERADOS = {(0, 1): 100, (1, 2): 0, (3, 4): 100, (4, 5): 0, (0, 3): 100, (1, 4): 100, (2, 5): 100}

# 1 + la mayor suma de capacidades que salen de un pixel (los vertices 0, 1, 3 y 4 suman 200)
CAPACIDAD_SEMILLA_ESPERADA = 201

def crear_grafo_con_n_links():
  alto, ancho = IMAGEN_TEST_GRAFO.shape
  grafo = nx.DiGraph()
  grafo.add_nodes_from(range(alto * ancho))
  agregar_n_links(grafo, IMAGEN_TEST_GRAFO, 30)
  return grafo

def verificar_penalidad(nombre):
  intensidad_p, intensidad_q, sigma, esperada = PENALIDADES_ESPERADAS[nombre]
  obtenida = penalidad_frontera(intensidad_p, intensidad_q, sigma)

  assert obtenida == esperada, f"[{nombre}] se esperaba {esperada} pero se obtuvo {obtenida}"
  print(f"OK [{nombre}] penalidad = {obtenida}")

def verificar_n_links():
  grafo = crear_grafo_con_n_links()

  for (vertice_a, vertice_b), esperada in N_LINKS_ESPERADOS.items():
    for origen, destino in ((vertice_a, vertice_b), (vertice_b, vertice_a)):
      assert grafo.has_edge(origen, destino), f"falta la arista {origen} -> {destino}"
      obtenida = grafo[origen][destino]["capacity"]
      assert obtenida == esperada, f"la arista {origen} -> {destino} tiene capacidad {obtenida} y se esperaba {esperada}"

  assert grafo.number_of_edges() == 2 * len(N_LINKS_ESPERADOS), f"se esperaban {2 * len(N_LINKS_ESPERADOS)} aristas pero hay {grafo.number_of_edges()}"
  print(f"OK n-links: {grafo.number_of_edges()} aristas")

def verificar_t_links():
  grafo = crear_grafo_con_n_links()
  n_links = grafo.number_of_edges()
  agregar_t_links(grafo, SEMILLAS_FONDO_TEST_GRAFO, SEMILLAS_OBJETO_TEST_GRAFO)

  for vertice in (0, 3):
    assert grafo.has_edge(FUENTE, vertice), f"falta la arista FUENTE -> {vertice} de la semilla de fondo"
    obtenida = grafo[FUENTE][vertice]["capacity"]
    assert obtenida == CAPACIDAD_SEMILLA_ESPERADA, f"FUENTE -> {vertice} tiene capacidad {obtenida} y se esperaba {CAPACIDAD_SEMILLA_ESPERADA}"

  for vertice in (2, 5):
    assert grafo.has_edge(vertice, SUMIDERO), f"falta la arista {vertice} -> SUMIDERO de la semilla de objeto"
    obtenida = grafo[vertice][SUMIDERO]["capacity"]
    assert obtenida == CAPACIDAD_SEMILLA_ESPERADA, f"{vertice} -> SUMIDERO tiene capacidad {obtenida} y se esperaba {CAPACIDAD_SEMILLA_ESPERADA}"

  assert grafo.in_degree(FUENTE) == 0, "no tiene que haber aristas que lleguen a la FUENTE"
  assert grafo.out_degree(SUMIDERO) == 0, "no tiene que haber aristas que salgan del SUMIDERO"
  assert grafo.number_of_edges() == n_links + 4, f"se esperaban {n_links + 4} aristas pero hay {grafo.number_of_edges()}"
  print(f"OK t-links: {grafo.number_of_edges() - n_links} aristas")

### Verificacion penalidad de frontera

In [ ]:
for nombre in PENALIDADES_ESPERADAS:
  verificar_penalidad(nombre)

### Verificacion n-links

In [ ]:
verificar_n_links()

### Verificacion t-links

In [ ]:
verificar_t_links()

### ***Ejercicio***: Calcular el corte mínimo

Completar `calcular_corte_minimo`, que dado el grafo devuelve el valor del corte y los dos conjuntos de vértices. Busquen en la [documentación de networkx](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.flow.minimum_cut.html) como utilizar la funcion `minimum_cut`.

Pueden inspeccionar el resultado sobre `grafo_chico` antes de seguir: ¿qué hay dentro de cada conjunto? ¿Aparecen `FUENTE` y `SUMIDERO`?

In [ ]:
def calcular_corte_minimo(grafo):
  # Devuelve (valor_del_corte, conjunto_fuente, conjunto_sumidero)
  pass

In [ ]:
valor_corte, conjunto_fuente, conjunto_sumidero = calcular_corte_minimo(grafo_chico)

print(f"Valor del corte: {valor_corte}")
print(f"Lado de la fuente:   {sorted(conjunto_fuente, key=str)}")
print(f"Lado del sumidero:   {sorted(conjunto_sumidero, key=str)}")

### Tests (corte mínimo)

In [ ]:
def crear_red_cadena():
  # Un unico cuello de botella: el corte minimo tiene que pasar por la arista a -> b
  red = nx.DiGraph()
  red.add_edge("s", "a", capacity=5)
  red.add_edge("a", "b", capacity=1)
  red.add_edge("b", "t", capacity=5)
  return red

# nombre: (red, valor esperado, lado de la fuente esperado). En la red del teorema hay varios cortes minimos, asi que solo se verifica que sea valido
REDES_CORTE_MINIMO = {
  'cadena':  (crear_red_cadena(), 1, {FUENTE, "a"}),
  'teorema': (red, 5, None),
}

def verificar_corte_minimo(nombre):
  red_test, valor_esperado, conjunto_fuente_esperado = REDES_CORTE_MINIMO[nombre]
  valor_corte, conjunto_fuente, conjunto_sumidero = calcular_corte_minimo(red_test)
  conjunto_fuente, conjunto_sumidero = set(conjunto_fuente), set(conjunto_sumidero)

  assert valor_corte == valor_esperado, f"[{nombre}] se esperaba un corte de {valor_esperado} pero se obtuvo {valor_corte}"
  assert FUENTE in conjunto_fuente and SUMIDERO in conjunto_sumidero, f"[{nombre}] la FUENTE tiene que estar en el primer conjunto y el SUMIDERO en el segundo"
  assert not (conjunto_fuente & conjunto_sumidero) and (conjunto_fuente | conjunto_sumidero) == set(red_test.nodes), f"[{nombre}] los conjuntos tienen que ser una particion de los vertices"
  assert capacidad_del_corte(red_test, conjunto_fuente) == valor_corte, f"[{nombre}] la capacidad de los conjuntos devueltos no coincide con el valor del corte"
  if conjunto_fuente_esperado is not None:
    assert conjunto_fuente == conjunto_fuente_esperado, f"[{nombre}] se esperaba {sorted(conjunto_fuente_esperado)} del lado de la fuente pero se obtuvo {sorted(conjunto_fuente)}"
  print(f"OK [{nombre}] corte de {valor_corte}")

### Verificacion corte mínimo

In [ ]:
for nombre in REDES_CORTE_MINIMO:
  verificar_corte_minimo(nombre)

### ***Ejercicio***: Interpretar el corte para armar la segmentación

El solver nos devuelve vértices, no una imagen. Completar `construir_segmentacion`, que dado el lado del sumidero (el objeto) y la forma de la imagen devuelve una **máscara booleana** con `True` en los píxeles de objeto.

Ayudas:

- Los vértices de los píxeles son enteros y `SUMIDERO` es un string: ¿qué hay que hacer con él?
- Los vértices están numerados por filas. ¿Cómo se vuelve de un vector de `alto * ancho` posiciones a una matriz de `alto x ancho`?

In [ ]:
def construir_segmentacion(conjunto_sumidero, forma):
  # El lado del sumidero es el objeto. Devuelve una mascara booleana de forma (alto, ancho)
  alto, ancho = forma
  segmentacion = np.zeros(alto * ancho, dtype=bool)

  # Descartamos el SUMIDERO: no es ningun pixel
  ids_pixeles_objeto = [vertice for vertice in conjunto_sumidero if vertice != SUMIDERO]
  segmentacion[ids_pixeles_objeto] = True

  # Como numeramos por filas, un reshape devuelve cada pixel a su lugar
  return segmentacion.reshape(forma)

### Tests (segmentación)

In [ ]:
# nombre: (lado del sumidero, forma de la imagen, segmentacion esperada). Los vertices se numeran por filas
SEGMENTACIONES_DESDE_CORTE = {
  'franja_derecha': ({2, 5, SUMIDERO}, (2, 3), np.array([[False, False, True], [False, False, True]])),
  'un_pixel':       ({SUMIDERO, 4},    (3, 3), np.array([[False, False, False], [False, True, False], [False, False, False]])),
  'sin_objeto':     ({SUMIDERO},       (2, 2), np.zeros((2, 2), dtype=bool)),
}

def verificar_construir_segmentacion(nombre):
  conjunto_sumidero, forma, esperada = SEGMENTACIONES_DESDE_CORTE[nombre]
  obtenida = construir_segmentacion(conjunto_sumidero, forma)

  assert obtenida.shape == forma, f"[{nombre}] se esperaba una matriz de forma {forma} pero se obtuvo {obtenida.shape}"
  assert obtenida.dtype == bool, f"[{nombre}] se esperaba una mascara booleana pero el tipo es {obtenida.dtype}"
  assert np.array_equal(obtenida, esperada), f"[{nombre}] se esperaba {esperada.tolist()} pero se obtuvo {obtenida.tolist()}"
  print(f"OK [{nombre}] {int(obtenida.sum())} pixeles de objeto")

### Verificacion segmentación

In [ ]:
for nombre in SEGMENTACIONES_DESDE_CORTE:
  verificar_construir_segmentacion(nombre)

Con todas las piezas armamos la segmentación completa.

In [ ]:
def segmentar_con_grafos(imagen, semillas_fondo, semillas_objeto, sigma=30):
  grafo = construir_grafo(imagen, semillas_fondo, semillas_objeto, sigma)
  valor_corte, conjunto_fuente, conjunto_sumidero = calcular_corte_minimo(grafo)
  print(f"Capacidad del corte minimo: {valor_corte}")
  return construir_segmentacion(conjunto_sumidero, imagen.shape)

### Tests (segmentación completa)

In [ ]:
# Imagenes donde se puede calcular a mano cual es el unico corte minimo
IMAGENES_SEGMENTACION = {
  'una_frontera':  np.array([[0, 0, 100, 100]], dtype=np.uint8),  # cortar entre 1 y 2 cuesta 3, cualquier otro corte cuesta 100
  'mitad_y_mitad': np.hstack((np.full((20, 10), 95), np.full((20, 10), 190))).astype(np.uint8),
}

# nombre: (rectangulos de fondo, rectangulos de objeto)
SEMILLAS_SEGMENTACION = {
  'una_frontera':  ([(0, 1, 0, 1)],  [(0, 1, 3, 4)]),
  'mitad_y_mitad': ([(0, 20, 0, 1)], [(0, 20, 19, 20)]),
}

SEGMENTACIONES_ESPERADAS = {
  'una_frontera':  np.array([[False, False, True, True]]),
  'mitad_y_mitad': np.tile(np.arange(20) >= 10, (20, 1)),
}

def verificar_segmentacion(nombre):
  imagen = IMAGENES_SEGMENTACION[nombre]
  rectangulos_fondo, rectangulos_objeto = SEMILLAS_SEGMENTACION[nombre]
  esperada = SEGMENTACIONES_ESPERADAS[nombre]
  inicio = time.perf_counter()
  obtenida = segmentar_con_grafos(imagen, crear_semillas(imagen.shape, rectangulos_fondo), crear_semillas(imagen.shape, rectangulos_objeto))
  tiempo = time.perf_counter() - inicio

  assert obtenida.shape == esperada.shape and np.array_equal(obtenida, esperada), f"[{nombre}] se esperaba {esperada.astype(int).tolist()} pero se obtuvo {obtenida.astype(int).tolist()}"
  print(f"OK [{nombre}]: {tiempo * 1000:.1f} ms")

def verificar_mascara_con_semillas(segmentacion, imagen, semillas_fondo, semillas_objeto):
  assert segmentacion.shape == imagen.shape and segmentacion.dtype == bool, "la segmentacion tiene que ser una mascara booleana del tamaño de la imagen"
  assert segmentacion[semillas_objeto].all(), "todas las semillas de objeto tienen que quedar dentro del objeto"
  assert not segmentacion[semillas_fondo].any(), "ninguna semilla de fondo puede quedar dentro del objeto"

CANTIDAD_MONEDAS = 8
# La imagen de monedas no tiene ruido, asi que Otsu tambien la segmenta bien y sirve de referencia
INTERSECCION_SOBRE_UNION_MINIMA_CON_OTSU = 0.9

def verificar_segmentacion_monedas():
  inicio = time.perf_counter()
  segmentacion = segmentar_con_grafos(imagen_monedas, SEMILLAS_FONDO, SEMILLAS_OBJETO)
  tiempo = time.perf_counter() - inicio

  verificar_mascara_con_semillas(segmentacion, imagen_monedas, SEMILLAS_FONDO, SEMILLAS_OBJETO)

  _, monedas_obtenidas = etiquetar_componentes(segmentacion)
  assert monedas_obtenidas == CANTIDAD_MONEDAS, f"se esperaban {CANTIDAD_MONEDAS} monedas separadas pero se obtuvieron {monedas_obtenidas}"

  segmentacion_otsu = imagen_monedas > otsu_manual(imagen_monedas)
  interseccion_sobre_union = (segmentacion & segmentacion_otsu).sum() / (segmentacion | segmentacion_otsu).sum()
  assert interseccion_sobre_union >= INTERSECCION_SOBRE_UNION_MINIMA_CON_OTSU, f"la segmentacion se parece demasiado poco a la de Otsu: {interseccion_sobre_union:.2f}"
  print(f"OK [monedas]: {monedas_obtenidas} monedas, parecido con Otsu {interseccion_sobre_union:.2f}, {tiempo * 1000:.1f} ms")

### Verificacion segmentación completa

In [ ]:
for nombre in IMAGENES_SEGMENTACION:
  verificar_segmentacion(nombre)

In [ ]:
verificar_segmentacion_monedas()

#### **Ejemplos**

In [ ]:
segmentacion_monedas = segmentar_con_grafos(imagen_monedas, SEMILLAS_FONDO, SEMILLAS_OBJETO)
mostrar_segmentacion(imagen_monedas, SEMILLAS_FONDO, SEMILLAS_OBJETO, segmentacion_monedas)

Probamos con otra imagen: el camarógrafo de `skimage`. Ahora el objeto es una sola figura de forma irregular, con el saco, la cabeza, la cámara y las piernas, y el fondo tiene cielo y pasto. Las semillas le indican al algoritmo qué queremos separar.

In [ ]:
imagen_camarografo = util.img_as_ubyte(transform.resize(data.camera(), (64, 64), anti_aliasing=True))

# Una mancha en el saco, otra en la cabeza, otra en la camara y otra en las piernas
SEMILLAS_OBJETO_CAMAROGRAFO = crear_semillas(imagen_camarografo.shape, [(30, 46, 3, 9), (14, 19, 16, 21), (24, 30, 23, 29), (52, 60, 9, 14)])

# El cielo, el pasto de abajo y el de la derecha
SEMILLAS_FONDO_CAMAROGRAFO = crear_semillas(imagen_camarografo.shape, [(0, 8, 0, 64), (50, 58, 34, 62), (24, 44, 52, 62)])

segmentacion_camarografo = segmentar_con_grafos(imagen_camarografo, SEMILLAS_FONDO_CAMAROGRAFO, SEMILLAS_OBJETO_CAMAROGRAFO)
mostrar_segmentacion(imagen_camarografo, SEMILLAS_FONDO_CAMAROGRAFO, SEMILLAS_OBJETO_CAMAROGRAFO, segmentacion_camarografo)

### Tests

In [ ]:
# El camarografo es la parte oscura de la imagen, asi que Otsu sirve de referencia aproximada (no coincide exacto)
INTERSECCION_SOBRE_UNION_MINIMA_CAMAROGRAFO = 0.8

def verificar_segmentacion_camarografo():
  inicio = time.perf_counter()
  segmentacion = segmentar_con_grafos(imagen_camarografo, SEMILLAS_FONDO_CAMAROGRAFO, SEMILLAS_OBJETO_CAMAROGRAFO)
  tiempo = time.perf_counter() - inicio

  verificar_mascara_con_semillas(segmentacion, imagen_camarografo, SEMILLAS_FONDO_CAMAROGRAFO, SEMILLAS_OBJETO_CAMAROGRAFO)

  # Las cuatro semillas de objeto estan en partes distintas del cuerpo: si quedan unidas es una sola figura
  _, figuras_obtenidas = etiquetar_componentes(segmentacion)
  assert figuras_obtenidas == 1, f"se esperaba una sola figura pero se obtuvieron {figuras_obtenidas}"

  segmentacion_oscura = imagen_camarografo <= otsu_manual(imagen_camarografo)
  interseccion_sobre_union = (segmentacion & segmentacion_oscura).sum() / (segmentacion | segmentacion_oscura).sum()
  assert interseccion_sobre_union >= INTERSECCION_SOBRE_UNION_MINIMA_CAMAROGRAFO, f"la segmentacion no sigue los bordes del camarografo: parecido con la parte oscura {interseccion_sobre_union:.2f}"
  print(f"OK [camarografo]: una figura de {int(segmentacion.sum())} pixeles, parecido con la parte oscura {interseccion_sobre_union:.2f}, {tiempo * 1000:.1f} ms")

verificar_segmentacion_camarografo()

#### **Comparación contra Otsu con una imagen ruidosa**

Umbralizar mira cada píxel por separado, así que el ruido se lo lleva puesto. El corte de grafo, en cambio, tiene que separar *regiones conectadas*: un píxel aislado que se parece a su entorno es caro de cortar.

Agregamos ruido a la imagen de monedas y comparamos. Prueben distintos valores de $\sigma$: si es muy chico el ruido empieza a parecer bordes y las monedas se pegan entre sí, y si es muy grande los bordes suaves dejan de contar como bordes y la segmentación se queda cerca de las semillas.

In [ ]:
generador_ruido = np.random.default_rng(0)
imagen_monedas_ruidosa = np.clip(imagen_monedas + generador_ruido.normal(0, 25, imagen_monedas.shape), 0, 255).astype(np.uint8)

T_otsu_ruidosa = otsu_manual(imagen_monedas_ruidosa)

def comparar_con_otsu(sigma):
  segmentacion_grafo = segmentar_con_grafos(imagen_monedas_ruidosa, SEMILLAS_FONDO, SEMILLAS_OBJETO, sigma)

  fig, ax = plt.subplots(1, 3, figsize=(15, 5))

  dibujar_semillas(ax[0], imagen_monedas_ruidosa, SEMILLAS_FONDO, SEMILLAS_OBJETO)

  ax[1].imshow(imagen_monedas_ruidosa > T_otsu_ruidosa, cmap="gray")
  ax[1].set_title(f"Otsu (T={T_otsu_ruidosa})")
  ax[1].axis("off")

  ax[2].imshow(segmentacion_grafo, cmap="gray")
  ax[2].set_title(f"Graph cut (sigma={sigma})")
  ax[2].axis("off")

  plt.tight_layout()
  plt.show()

# El corte tarda cerca de un segundo, por eso solo se recalcula al soltar el slider
interact(
  comparar_con_otsu,
  sigma=IntSlider(
    min=10,
    max=80,
    step=5,
    value=35,
    description='sigma',
    continuous_update=False
  )
);